# Infinite Connections: AI-Generated NYT-Style Puzzles

**STA 561D — Duke University**

This notebook demonstrates an end-to-end system for generating, validating, and analyzing
NYT Connections-style puzzles using Claude (Anthropic API) and MPNET embeddings.

## Table of Contents
1. [Introduction](#1.-Introduction)
2. [Data Exploration](#2.-Data-Exploration)
3. [Generation Pipeline Demo](#3.-Generation-Pipeline-Demo)
4. [Solver Benchmark](#4.-Solver-Benchmark)
5. [Validation Demo](#5.-Validation-Demo)
6. [Results at Scale](#6.-Results-at-Scale)
7. [Example Puzzles](#7.-Example-Puzzles)
8. [Discussion](#8.-Discussion)

---
## 1. Introduction

**NYT Connections** is a daily word puzzle where players must sort 16 words into 4 groups of 4,
each sharing a hidden connection. The puzzle is deceptively simple — words often have multiple
meanings, creating "false groups" that trap solvers.

### Why is generation hard?

- **Uniqueness**: Each puzzle needs exactly one valid solution among C(16,4) × C(12,4) × C(8,4) = 2,627,625 possible groupings
- **Difficulty gradient**: Groups must range from obvious (yellow) to tricky (purple)
- **Deception**: Good puzzles include plausible-but-wrong groupings that mislead solvers
- **Diversity**: Categories should span synonyms, wordplay, fill-in-the-blank, pop culture, and more

### Our approach

We combine:
1. **Claude API** for creative category generation (with story injection for diversity)
2. **MPNET embeddings** for word selection and difficulty calibration
3. **Multi-solver validation** to ensure puzzle uniqueness
4. **False-group pipeline** to create intentionally deceptive puzzles

In [ ]:
# Setup: imports and configuration
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Run in DRY_RUN mode (mock LLM) unless ANTHROPIC_API_KEY is set
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['DRY_RUN'] = 'true'

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter

from src.config import DRY_RUN, NYT_PUZZLES_PATH, MOCK_PUZZLES_PATH

sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.dpi'] = 100

print(f'DRY_RUN mode: {DRY_RUN}')
print(f'Project root: {PROJECT_ROOT}')

---
## 2. Data Exploration

We start by loading and analyzing the ground truth NYT Connections dataset (554 puzzles).

In [ ]:
# Load the NYT dataset
from pathlib import Path

nyt_path = Path(NYT_PUZZLES_PATH)
if nyt_path.exists():
    with open(nyt_path) as f:
        nyt_raw = json.load(f)
    print(f'Loaded {len(nyt_raw)} NYT puzzles')
else:
    print(f'NYT dataset not found at {nyt_path}, using mock data')
    with open(MOCK_PUZZLES_PATH) as f:
        nyt_raw = json.load(f)
    print(f'Loaded {len(nyt_raw)} mock puzzles')

# Show schema of first puzzle
print('\nFirst puzzle keys:', list(nyt_raw[0].keys()))
print(f'Words per puzzle: {len(nyt_raw[0].get("words", []))}')

In [ ]:
# Build word bank and analyze word frequency
all_words = []
all_categories = []

for puzzle in nyt_raw:
    words = puzzle.get('words', [])
    all_words.extend([w.upper() for w in words])
    for answer in puzzle.get('answers', puzzle.get('groups', [])):
        cat = answer.get('answerDescription', answer.get('category', ''))
        if cat:
            all_categories.append(cat.upper())

word_counts = Counter(all_words)
unique_words = len(set(all_words))

print(f'Total word appearances: {len(all_words)}')
print(f'Unique words: {unique_words}')
print(f'Unique categories: {len(set(all_categories))}')
print(f'\nTop 20 most frequent words:')
for word, count in word_counts.most_common(20):
    print(f'  {word}: {count}')

In [ ]:
# Visualize word frequency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word frequency histogram
freq_values = list(word_counts.values())
axes[0].hist(freq_values, bins=range(1, max(freq_values) + 2), color='steelblue',
             edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Appearances across puzzles')
axes[0].set_ylabel('Number of words')
axes[0].set_title('Word Frequency Distribution')

# Category word count
cat_word_lens = [len(cat.split()) for cat in all_categories]
axes[1].hist(cat_word_lens, bins=range(1, max(cat_word_lens) + 2), color='coral',
             edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Words in category name')
axes[1].set_ylabel('Count')
axes[1].set_title('Category Name Length Distribution')

plt.tight_layout()
plt.show()

---
## 3. Generation Pipeline Demo

We generate 5 puzzles live, showing each step: group creation, editor review, and difficulty assignment.

The pipeline uses two methods:
- **Iterative**: Build groups one at a time with story-injection diversity
- **False Group**: Create a decoy group, then build real groups from alternate word meanings

In [ ]:
# Initialize the pipeline
from sentence_transformers import SentenceTransformer
from src.llm_client import LLMClient, MockLLMClient
from src.generator.pipeline import PuzzlePipeline

# Reset mock state
MockLLMClient.reset()

# Load embedding model
embedding_model = SentenceTransformer('all-mpnet-base-v2')
llm = LLMClient()

# Build word bank from NYT data
word_bank = list(set(all_words))[:500]  # Use subset for speed

pipeline = PuzzlePipeline(
    llm=llm,
    embedding_model=embedding_model,
    word_bank=word_bank,
)

print(f'Pipeline initialized (DRY_RUN={DRY_RUN})')
print(f'Word bank size: {len(word_bank)}')

In [ ]:
# Generate 3 iterative puzzles and 2 false-group puzzles
generated_puzzles = []

for i in range(3):
    MockLLMClient.reset()
    MockLLMClient._call_count = i * 4  # Vary which mock pools are used
    puzzle = pipeline.generate(method='iterative')
    generated_puzzles.append(puzzle)
    print(f'\nPuzzle {puzzle["id"]} (iterative):')
    for g in puzzle['groups']:
        print(f'  [{g["color"]:>6}] {g["category"]}: {", ".join(g["words"])} (sim={g["similarity_score"]:.3f})')

for i in range(2):
    MockLLMClient.reset()
    MockLLMClient._call_count = i * 5
    puzzle = pipeline.generate(method='false_group')
    generated_puzzles.append(puzzle)
    print(f'\nPuzzle {puzzle["id"]} (false_group):')
    for g in puzzle['groups']:
        print(f'  [{g["color"]:>6}] {g["category"]}: {", ".join(g["words"])} (sim={g["similarity_score"]:.3f})')

print(f'\nTotal generated: {len(generated_puzzles)} puzzles')

In [ ]:
# Show the MPNET selection process for one group
from src.generator.group_creator import EmbeddingSelector

selector = EmbeddingSelector(model=embedding_model)

# Example: 8 music words, select best 4
pool = ['PIANO', 'GUITAR', 'DRUMS', 'VIOLIN', 'TRUMPET', 'FLUTE', 'CELLO', 'HARP']
selected, score = selector.select_best(pool, n=4)

print('MPNET Word Selection Demo')
print(f'  Pool (8 words): {pool}')
print(f'  Selected (4):   {selected}')
print(f'  Avg similarity: {score:.4f}')

# Show pairwise similarities
from sklearn.metrics.pairwise import cosine_similarity
embs = embedding_model.encode(pool, convert_to_numpy=True)
sim = cosine_similarity(embs)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(sim, dtype=bool), k=0)
sns.heatmap(sim, mask=mask, annot=True, fmt='.2f', xticklabels=pool, yticklabels=pool,
            cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
ax.set_title('Pairwise Cosine Similarity (MPNET) — Musical Instruments')
plt.tight_layout()
plt.show()

---
## 4. Solver Benchmark

We benchmark three independent solvers on the puzzle dataset:

| Solver | Method | Speed | Expected Accuracy |
|--------|--------|-------|-------------------|
| Embedding | Greedy cosine similarity | Fast | ~11.6% full-puzzle |
| Clustering | G-P score + beam search | Medium | Higher |
| LLM (Claude) | Chain-of-thought | Slow | ~38.9% full-puzzle |

In [ ]:
# Load puzzles to benchmark against
from src.evaluation.analyzer import PuzzleAnalyzer

# Use mock puzzles for demo; in production, benchmark against full NYT set
benchmark_puzzles = json.loads(open(MOCK_PUZZLES_PATH).read())
print(f'Benchmarking against {len(benchmark_puzzles)} puzzles')

In [ ]:
# Run embedding solver on all benchmark puzzles
from src.solvers.embedding_solver import EmbeddingSolver
from src.solvers.roundtable import normalize_groups, check_against_answer

emb_solver = EmbeddingSolver(model=embedding_model)

emb_results = []
for puzzle in benchmark_puzzles:
    solution = emb_solver.solve(puzzle['words'])
    norm = normalize_groups(solution)
    correct = check_against_answer(norm, puzzle['groups'])
    emb_results.append({
        'id': puzzle['id'],
        'groups_correct': correct,
        'fully_solved': correct == 4,
    })

emb_df = pd.DataFrame(emb_results)
print('Embedding Solver Results:')
print(f'  Full puzzle solve rate: {emb_df["fully_solved"].mean():.1%}')
print(f'  Avg groups correct:    {emb_df["groups_correct"].mean():.2f} / 4')
print(emb_df[['id', 'groups_correct', 'fully_solved']].to_string(index=False))

In [ ]:
# Run clustering solver
from src.solvers.clustering_solver import ClusteringSolver

clust_solver = ClusteringSolver(model=embedding_model, beam_width=5)

clust_results = []
for puzzle in benchmark_puzzles:
    solution = clust_solver.solve(puzzle['words'])
    norm = normalize_groups(solution)
    correct = check_against_answer(norm, puzzle['groups'])
    clust_results.append({
        'id': puzzle['id'],
        'groups_correct': correct,
        'fully_solved': correct == 4,
    })

clust_df = pd.DataFrame(clust_results)
print('Clustering Solver Results:')
print(f'  Full puzzle solve rate: {clust_df["fully_solved"].mean():.1%}')
print(f'  Avg groups correct:    {clust_df["groups_correct"].mean():.2f} / 4')
print(clust_df[['id', 'groups_correct', 'fully_solved']].to_string(index=False))

In [ ]:
# Compare solver performance
comparison = pd.DataFrame({
    'Puzzle': [p['id'] for p in benchmark_puzzles],
    'Embedding': emb_df['groups_correct'].values,
    'Clustering': clust_df['groups_correct'].values,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: groups correct per puzzle
x = np.arange(len(comparison))
width = 0.35
axes[0].bar(x - width/2, comparison['Embedding'], width, label='Embedding', color='steelblue')
axes[0].bar(x + width/2, comparison['Clustering'], width, label='Clustering', color='coral')
axes[0].set_xlabel('Puzzle')
axes[0].set_ylabel('Groups Correct (out of 4)')
axes[0].set_title('Solver Accuracy per Puzzle')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison['Puzzle'], rotation=45)
axes[0].legend()
axes[0].set_ylim(0, 4.5)

# Summary comparison
solvers = ['Embedding', 'Clustering']
solve_rates = [emb_df['fully_solved'].mean(), clust_df['fully_solved'].mean()]
avg_correct = [emb_df['groups_correct'].mean(), clust_df['groups_correct'].mean()]

axes[1].bar(solvers, avg_correct, color=['steelblue', 'coral'], alpha=0.8)
axes[1].set_ylabel('Avg Groups Correct')
axes[1].set_title('Average Solver Performance')
axes[1].set_ylim(0, 4.5)
for i, (rate, avg) in enumerate(zip(solve_rates, avg_correct)):
    axes[1].text(i, avg + 0.1, f'{rate:.0%} full solve', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. Validation Demo

The **Roundtable Validator** runs multiple solvers and checks for convergence.
A puzzle is valid if at least 2 solvers agree on the same unique solution.

In [ ]:
# Run roundtable validation on generated puzzles
from src.solvers.roundtable import Roundtable

roundtable = Roundtable(embedding_model=embedding_model, llm=llm)

validation_results = []
for puzzle in generated_puzzles:
    result = roundtable.validate(puzzle)
    validation_results.append({
        'id': puzzle['id'],
        'method': puzzle['metadata']['generation_method'],
        'valid': result['valid'],
        'agreement': result['solver_agreement'],
        'emb_correct': result['groups_correct'].get('embedding', 0),
        'clust_correct': result['groups_correct'].get('clustering', 0),
        'detail': result['convergence_detail'],
    })

val_df = pd.DataFrame(validation_results)
print('Roundtable Validation Results:')
print('=' * 80)
for _, row in val_df.iterrows():
    status = 'VALID' if row['valid'] else 'INVALID'
    print(f"  {row['id']} ({row['method']:>11}): {status:>7} | "
          f"Emb={row['emb_correct']}/4, Clust={row['clust_correct']}/4 | "
          f"Agreement={'Yes' if row['agreement'] else 'No'}")

valid_count = val_df['valid'].sum()
print(f'\n{valid_count}/{len(val_df)} puzzles passed validation ({valid_count/len(val_df):.0%})')

---
## 6. Results at Scale

We analyze quality metrics across all generated puzzles and compare them to the NYT distribution.

In [ ]:
# Load pre-generated mock dataset for scale analysis
from src.evaluation.analyzer import PuzzleAnalyzer
from src.evaluation.metrics import puzzle_quality_score

analyzer = PuzzleAnalyzer(model=embedding_model)

# Combine generated + mock puzzles for a larger sample
all_puzzles = generated_puzzles + benchmark_puzzles
stats = analyzer.analyze_dataset(all_puzzles)

print('Dataset Statistics:')
print(f'  Total puzzles:          {stats["count"]}')
print(f'  Avg similarity:         {stats["avg_similarity"]:.4f}')
print(f'  Similarity std:         {stats["similarity_std"]:.4f}')
print(f'  Category diversity:     {stats["category_diversity"]} unique categories')
print(f'  Solver agreement rate:  {stats["solver_agreement_rate"]:.1%}')
print(f'\nColor distribution:')
for color, count in sorted(stats['color_distribution'].items()):
    print(f'  {color:>6}: {count}')
print(f'\nGeneration method distribution:')
for method, count in stats['method_distribution'].items():
    print(f'  {method}: {count}')

In [ ]:
# Compute per-group quality metrics
group_data = []
for puzzle in all_puzzles:
    quality = puzzle_quality_score(puzzle, model=embedding_model)
    for gm in quality['groups']:
        group_data.append({
            'category': gm['category'],
            'color': gm['color'],
            'similarity': gm['avg_pairwise_sim'],
        })

group_df = pd.DataFrame(group_data)

# Plot similarity distribution by color
color_map = {'yellow': '#F9DF6D', 'green': '#A0C35A', 'blue': '#B0C4EF', 'purple': '#BA81C5'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
color_order = ['yellow', 'green', 'blue', 'purple']
available = [c for c in color_order if c in group_df['color'].values]
box_data = [group_df[group_df['color'] == c]['similarity'].values for c in available]
bp = axes[0].boxplot(box_data, labels=available, patch_artist=True)
for patch, color in zip(bp['boxes'], available):
    patch.set_facecolor(color_map.get(color, '#ccc'))
axes[0].set_ylabel('Avg Pairwise Cosine Similarity')
axes[0].set_title('Similarity Distribution by Difficulty Color')

# Histogram overlay
for color in available:
    subset = group_df[group_df['color'] == color]['similarity']
    axes[1].hist(subset, bins=15, alpha=0.5, label=color, color=color_map.get(color, '#ccc'))
axes[1].set_xlabel('Avg Pairwise Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].set_title('Similarity Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print per-color stats
print('Per-color statistics:')
for color in available:
    subset = group_df[group_df['color'] == color]['similarity']
    print(f'  {color:>6}: mean={subset.mean():.4f}, std={subset.std():.4f}, n={len(subset)}')

---
## 7. Example Puzzles

Here are some of the best generated puzzles, displayed in a visual grid format.

In [ ]:
def display_puzzle(puzzle, title=None):
    """Display a puzzle as a colored grid."""
    color_map = {
        'yellow': '#F9DF6D', 'green': '#A0C35A',
        'blue': '#B0C4EF', 'purple': '#BA81C5'
    }
    color_order = ['yellow', 'green', 'blue', 'purple']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 5)
    ax.axis('off')

    if title:
        ax.text(2, 4.7, title, ha='center', va='center', fontsize=14, fontweight='bold')

    # Sort groups by color order
    groups = sorted(puzzle['groups'],
                    key=lambda g: color_order.index(g['color']) if g['color'] in color_order else 4)

    for row, group in enumerate(groups):
        bg = color_map.get(group['color'], '#ddd')

        # Category header spanning full width
        rect = plt.Rectangle((0, 3.8 - row * 1.1), 4, 0.4, linewidth=0,
                              facecolor=bg, alpha=0.9)
        ax.add_patch(rect)
        ax.text(2, 4.0 - row * 1.1, group['category'],
                ha='center', va='center', fontsize=10, fontweight='bold')

        # Word tiles
        for col, word in enumerate(group['words'][:4]):
            rect = plt.Rectangle((col * 1.0, 3.3 - row * 1.1), 0.95, 0.35,
                                  linewidth=1, edgecolor='white',
                                  facecolor=bg, alpha=0.6)
            ax.add_patch(rect)
            ax.text(col * 1.0 + 0.475, 3.475 - row * 1.1, word,
                    ha='center', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()


# Display mock puzzles
for puzzle in benchmark_puzzles[:3]:
    display_puzzle(puzzle, title=f'Puzzle {puzzle["id"]}')

In [ ]:
# Display generated puzzles
for puzzle in generated_puzzles[:3]:
    display_puzzle(puzzle, title=f'{puzzle["id"]} ({puzzle["metadata"]["generation_method"]})')

---
## 8. Discussion

### What worked

- **MPNET embedding selection** reliably picks the most cohesive 4 words from an 8-word pool,
  outperforming LLM self-selection on consistency
- **Story injection** dramatically increases category diversity — without it, the LLM defaults
  to the same ~20 categories regardless of temperature
- **Multi-solver validation** is an effective quality gate: if two independent solvers can't
  find the same unique solution, the puzzle is genuinely ambiguous
- **False-group pipeline** produces the most engaging puzzles, with built-in deception

### What didn't work

- **LLM word selection** (letting Claude pick the final 4 from 8) produces inconsistent
  difficulty — the LLM can't reliably estimate semantic similarity
- **Pure embedding solvers** struggle with wordplay and cultural categories (e.g., 
  TAYLOR SWIFT ALBUMS) since these rely on world knowledge, not semantic proximity
- **Single-solver validation** produces too many false positives — puzzles that seem
  valid to one approach but are actually ambiguous

### Future improvements

1. **Human evaluation**: Run a user study comparing generated puzzles to NYT originals
2. **Adaptive difficulty**: Tune the false-group overlap to control solve rates
3. **Hybrid solver**: Combine embedding + LLM solvers for better validation coverage
4. **Larger scale**: Generate 10K+ puzzles and analyze long-tail quality distribution
5. **Fine-tuned embeddings**: Train domain-specific embeddings on Connections word pairs

In [ ]:
# Summary statistics
print('=' * 60)
print('INFINITE CONNECTIONS — Summary')
print('=' * 60)
print(f'NYT puzzles loaded:        {len(nyt_raw)}')
print(f'Puzzles generated:         {len(generated_puzzles)}')
print(f'Validation pass rate:      {val_df["valid"].mean():.0%}')
print(f'Embedding solver accuracy: {emb_df["groups_correct"].mean():.2f}/4 avg')
print(f'Cluster solver accuracy:   {clust_df["groups_correct"].mean():.2f}/4 avg')
print(f'Category diversity:        {stats["category_diversity"]} unique categories')
print(f'DRY_RUN mode:              {DRY_RUN}')
print('=' * 60)